## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/delhi/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/delhi/


In [7]:
##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Delhi'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Delhi, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [8]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(1046336, 2)


,customer_id,usecase_tag
0,63b2d1c018d96a850388b280,Unknown
1,64d0cfab53729c4fd006a332,Unknown


### Active customer (RR-customers)

In [9]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/delhi_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(957065, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,60412a8aad5f34e06a2e3c62,MALE,1129.0,28.0,20.0,11.0,6.0,PHH,70,HOOK,HIGH_INCOME,NO_AFFINITY,LONG_DISTANCE,NPS,6.0,d. high rpc
1,605f349e67271d030ae4e7c5,MALE,1107.0,25.0,15.0,8.0,5.0,PHH,14,SUSTENANCE,MEDIUM_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,5.0,d. high rpc


In [10]:
df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
                                        how='left', on=['customer_id'])

### customer installed apps

In [11]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(956413, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,573f28f39b0ffc283676f2b3,"['telegram', 'ludo king', 'instagram', 'zoom',...",48,"['Tools', 'Social', 'Educational', 'News', 'De...",17,"['Educational', 'Gaming', 'Office', 'Finance_I...",6,0,1,1,1,1,1,1
1,573f28f49b0ffc283676f511,"['telegram', 'instagram', 'booking.com', 'cred...",55,"['Tools', 'Social', 'Educational', 'Delivery_F...",15,"['Educational', 'Gaming', 'Office', 'Finance_I...",6,0,1,1,1,1,1,1


### Customer app & cat explode mapping

In [12]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,60412a8aad5f34e06a2e3c62,Rest


In [13]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

956413

In [14]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Social' 'Driver_App' 'Streaming_Music' 'Finance_Transactions' 'Tools'
 'News' 'Travel_Ridehailing' 'Ecommerce' 'Finance_Investment'
 'Educational' 'Travel_Bookings' 'Delivery_Grocery' 'Office' 'OTT'
 'Gaming' 'Delivery_Food' 'Fantasy_Sports' 'Entertainment' 'Wearable'
 'Finance_News' 'Health' 'Dating' 'Devotional']


In [15]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Driver_App' 'Ride haling' 'Finance_Investment' 'Educational'
 'Office' 'Gaming']


In [16]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [18]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,uber,487502,50.97
1,Ride haling,ola,447034,46.74
2,Ride haling,in drive,141782,14.82
3,Ride haling,blusmart,55519,5.80
4,Ride haling,quick ride,6387,0.67
5,Ride haling,namma yatri,6024,0.63
6,Ride haling,jugnoo,1336,0.14
7,Ride haling,driveu driver app,961,0.10
8,Ride haling,quickride,257,0.03
9,Office,zoom,220941,23.10


### Merge raw data

In [19]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(956413, 30)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest
0,60412a8aad5f34e06a2e3c62,MALE,1129.0,28.0,20.0,11.0,6.0,PHH,70,HOOK,HIGH_INCOME,NO_AFFINITY,LONG_DISTANCE,NPS,6.0,d. high rpc,office,"['myntra', 'instagram', 'facebook', 'wynk musi...",14,"['Tools', 'Social', 'News', 'Ecommerce', 'Stre...",9,"['Driver_App', 'Ride haling', 'Finance_Investm...",4,1,0,0,1,0,1,1


#### Derived columns

In [20]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    752052.000000
mean          2.797648
std           3.437470
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          90.000000
Name: rpc, dtype: float64

In [21]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [22]:
## app_count_bucket

df_customer_data.app_count.describe()

count    956413.000000
mean         17.726033
std           9.052043
min           1.000000
25%          11.000000
50%          16.000000
75%          23.000000
max          93.000000
Name: app_count, dtype: float64

In [23]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [24]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    956413.000000
mean          3.063021
std           1.264304
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

In [25]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 4 , '2-3','Above 3'))

In [26]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    956096.000000
mean        589.066012
std         538.070392
min           1.000000
25%         135.000000
50%         473.000000
75%         848.000000
max        2876.000000
Name: rapido_age, dtype: float64

#### Merge

In [27]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '6475f56aba9a99b915ccc086'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,usecase_tag,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Finance_Investment,Office,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [28]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,955977,99.95
1,Ride haling,652741,68.25
2,Office,500778,52.36
3,Finance_Investment,356539,37.28
4,Gaming,193791,20.26
5,Educational,188309,19.69
6,Driver_App,81378,8.51


#### LTR-Segment

In [29]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,251958,26.34
1,LTR 0,53366,5.58
2,PHH,650659,68.03
3,UNKNOWN,430,0.04


In [30]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                 \
categories_l2 Driver_App Educational Finance_Investment  Gaming  Office   
ltr_segment                                                               
HH                 25474       42430              72760   51438  105438   
LTR 0               6581        7971              12543   11342   18649   
PHH                49297      137814             271073  130942  376420   

                                   
categories_l2    Rest Ride haling  
ltr_segment                        
HH             251791      139457  
LTR 0           53311       26981  
PHH            650445      485987

In [31]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [32]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                  10.11       16.84              28.88  20.42  41.85  99.93   
LTR 0               12.33       14.94              23.50  21.25  34.95  99.90   
PHH                  7.58       21.18              41.66  20.12  57.85  99.97   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  55.35  
LTR 0               50.56  
PHH                 74.69

#### Other testing

In [33]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [34]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,139457,21.36
1,LTR 0,26981,4.13
2,PHH,485987,74.45
3,UNKNOWN,316,0.05


In [35]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                        \
app_name     blusmart driveu driver app in drive jugnoo namma yatri     ola   
ltr_segment                                                                   
HH               7245               225    23962    179         818   90953   
LTR 0            1254                27     4825     35         161   17091   
PHH             46990               709   112948   1121        5045  338766   

                                          
app_name    quick ride quickride    uber  
ltr_segment                               
HH                 640        89   95537  
LTR 0              104        36   17575  
PHH               5639       132  374149

In [36]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [37]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 5.20              0.16    17.18   0.13        0.59  65.22   
LTR 0              4.65              0.10    17.88   0.13        0.60  63.34   
PHH                9.67              0.15    23.24   0.23        1.04  69.71   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                0.46      0.06  68.51  
LTR 0             0.39      0.13  65.14  
PHH               1.16      0.03  76.99

##### Explode - Office

In [38]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
3,605f349e67271d030ae4e7c5,onedrive,onedrive,Office,Office,PHH
4,64a04c91eaba57b5f7bdcd39,microsoft teams,microsoft teams,Office,Office,PHH


In [39]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
3,605f349e67271d030ae4e7c5,onedrive,onedrive,Office,Office,PHH,secondary_office_app
4,64a04c91eaba57b5f7bdcd39,microsoft teams,microsoft teams,Office,Office,PHH,secondary_office_app


In [40]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'slack', 'trello', 'github', 'zoho mail',
       'jira', 'asana', 'cisco', 'zoho meeting', 'google analytics',
       'miro', 'confluence'], dtype=object)

In [41]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,105438,21.05
1,LTR 0,18649,3.72
2,PHH,376420,75.17
3,UNKNOWN,271,0.05


In [42]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                     \
app_name        asana  cisco confluence dropbox  github google analytics   
ltr_segment                                                                
HH              113.0   52.0       43.0  1434.0   748.0            268.0   
LTR 0             9.0    8.0        5.0   230.0   100.0             45.0   
PHH             851.0  374.0      352.0  6557.0  4529.0           1504.0   

                                                                               \
app_name    google authenticator intune  company portal    jira microsoft 365   
ltr_segment                                                                     
HH                        7626.0                 6788.0   194.0       37112.0   
LTR 0                     1155.0                  924.0    24.0        6503.0   
PHH                      40534.0                43140.0  2038.0      139750.0   

                                                                       \
app_name    microsoft teams   miro ms authenticator nishtha  onedrive   
ltr_segment                                                             
HH                  27386.0   43.0          10551.0     1.0   50246.0   
LTR 0                4098.0    3.0           1397.0     NaN    9648.0   
PHH                139075.0  295.0          66216.0     2.0  155828.0   

                                                                               \
app_name      outlook pocket    slack  trello    webex zoho mail zoho meeting   
ltr_segment                                                                     
HH            35534.0  100.0   1926.0   143.0   4824.0     594.0        223.0   
LTR 0          6007.0   16.0    254.0    17.0    756.0      69.0         25.0   
PHH          147435.0  873.0  16819.0  1149.0  22086.0    3618.0       1033.0   

                       
app_name         zoom  
ltr_segment            
HH            44170.0  
LTR 0          7461.0  
PHH          169179.0

In [43]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [44]:
print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual app.


customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.11  0.05       0.04    1.36   0.71             0.25   
LTR 0              0.05  0.04       0.03    1.23   0.54             0.24   
PHH                0.23  0.10       0.09    1.74   1.20             0.40   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          7.23                   6.44  0.18         35.20   
LTR 0                       6.19                   4.95  0.13         34.87   
PHH                        10.77                  11.46  0.54         37.13   

                                                                             \
app_name    microsoft teams  miro ms authenticator nishtha onedrive outlook   
ltr_segment                                                                   
HH                    25.97  0.04            10.01     0.0    47.65   33.70   
LTR 0                 21.97  0.02             7.49     NaN    51.73   32.21   
PHH                   36.95  0.08            17.59     0.0    41.40   39.17   

                                                                     
app_name    pocket slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                          
HH            0.09  1.83   0.14  4.58      0.56         0.21  41.89  
LTR 0         0.09  1.36   0.09  4.05      0.37         0.13  40.01  
PHH           0.23  4.47   0.31  5.87      0.96         0.27  44.94

In [45]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                     10305               104477
LTR 0                   1377                18514
PHH                    68144               370741

In [46]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                      9.77                99.09
LTR 0                   7.38                99.28
PHH                    18.10                98.49

##### Explode - Education

In [47]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
1,605f349e67271d030ae4e7c5,physics wallah,physics wallah,Educational,Educational,PHH
19,65997f6e6d2e7674adffc2ea,udemy,udemy,Educational,Educational,PHH


In [48]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
1,605f349e67271d030ae4e7c5,physics wallah,physics wallah,Educational,Educational,PHH,code_education_app
19,65997f6e6d2e7674adffc2ea,udemy,udemy,Educational,Educational,PHH,secondary_education_app


In [49]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['physics wallah', 'simplilearn', 'byju', 'google classroom',
       'vidyakul', 'vedantu', 'e-pg pathshala', 'diksha', 'mycbseguide',
       'moodle', 'chegg study', 'aakash', 'microsoft math solver'],
      dtype=object)

In [50]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,42430,22.53
1,LTR 0,7971,4.23
2,PHH,137814,73.19
3,UNKNOWN,94,0.05


In [51]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                               \
app_name       aakash bansal classes  brainly     byju caclubindia   
ltr_segment                                                          
HH              218.0            NaN   5934.0   5435.0        13.0   
LTR 0            47.0            NaN   1151.0   1305.0         4.0   
PHH             413.0            2.0  15971.0  12331.0        81.0   

                                                                        \
app_name    chegg study colegeduniya collegedekho course hero coursera   
ltr_segment                                                              
HH                130.0         50.0          5.0         5.0   1931.0   
LTR 0              21.0          5.0          3.0         4.0    268.0   
PHH               452.0        154.0         15.0        36.0  12201.0   

                                                                             \
app_name     diksha doubtnut duolingo e-pg pathshala     edx embibe fiitjee   
ltr_segment                                                                   
HH           1003.0   1759.0   8131.0          372.0   265.0  162.0     1.0   
LTR 0         197.0    417.0   1544.0           66.0    37.0   34.0     NaN   
PHH          2252.0   3420.0  26670.0         1154.0  1734.0  438.0    10.0   

                                                                \
app_name    geeks for geeks goodreads google classroom happify   
ltr_segment                                                      
HH                    474.0     297.0           9584.0     NaN   
LTR 0                  62.0      62.0           1605.0     NaN   
PHH                  2776.0    2007.0          32475.0    19.0   

                                                                            \
app_name    ignou e-content khan academy meritnation microsoft math solver   
ltr_segment                                                                  
HH                    880.0        183.0         NaN                 138.0   
LTR 0                 160.0         36.0         NaN                  23.0   
PHH                  2741.0        837.0         3.0                 498.0   

                                                                     \
app_name     moodle my study life mycbseguide ncert books photomath   
ltr_segment                                                           
HH            343.0          13.0       835.0       986.0     257.0   
LTR 0          55.0           2.0       180.0       169.0      52.0   
PHH          1960.0          46.0      1640.0      2712.0     863.0   

                                                                         \
app_name    physics wallah pocket shiksha mitra shiksha.com simplilearn   
ltr_segment                                                               
HH                  6243.0  100.0          16.0        69.0       513.0   
LTR 0               1230.0   16.0           3.0        12.0        88.0   
PHH                14819.0  873.0          40.0       241.0      2952.0   

                                                                              \
app_name    stack overflow  swayam toppr    udemy unacademy vedantu vidyakul   
ltr_segment                                                                    
HH                     3.0  1402.0   5.0   4479.0    5196.0   230.0     41.0   
LTR 0                  NaN   220.0   NaN    577.0     950.0    50.0     16.0   
PHH                   23.0  5938.0  14.0  28514.0   16557.0   537.0     80.0   

                         
app_name    whitehat jr  
ltr_segment              
HH                121.0  
LTR 0              26.0  
PHH               271.0

In [52]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [53]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                        \
app_name         aakash bansal classes brainly   byju caclubindia chegg study   
ltr_segment                                                                     
HH                 0.51            NaN   13.99  12.81        0.03        0.31   
LTR 0              0.59            NaN   14.44  16.37        0.05        0.26   
PHH                0.30            0.0   11.59   8.95        0.06        0.33   

                                                                            \
app_name    colegeduniya collegedekho course hero coursera diksha doubtnut   
ltr_segment                                                                  
HH                  0.12         0.01        0.01     4.55   2.36     4.15   
LTR 0               0.06         0.04        0.05     3.36   2.47     5.23   
PHH                 0.11         0.01        0.03     8.85   1.63     2.48   

                                                                          \
app_name    duolingo e-pg pathshala   edx embibe fiitjee geeks for geeks   
ltr_segment                                                                
HH             19.16           0.88  0.62   0.38    0.00            1.12   
LTR 0          19.37           0.83  0.46   0.43     NaN            0.78   
PHH            19.35           0.84  1.26   0.32    0.01            2.01   

                                                                             \
app_name    goodreads google classroom happify ignou e-content khan academy   
ltr_segment                                                                   
HH               0.70            22.59     NaN            2.07         0.43   
LTR 0            0.78            20.14     NaN            2.01         0.45   
PHH              1.46            23.56    0.01            1.99         0.61   

                                                                    \
app_name    meritnation microsoft math solver moodle my study life   
ltr_segment                                                          
HH                  NaN                  0.33   0.81          0.03   
LTR 0               NaN                  0.29   0.69          0.03   
PHH                 0.0                  0.36   1.42          0.03   

                                                                     \
app_name    mycbseguide ncert books photomath physics wallah pocket   
ltr_segment                                                           
HH                 1.97        2.32      0.61          14.71   0.24   
LTR 0              2.26        2.12      0.65          15.43   0.20   
PHH                1.19        1.97      0.63          10.75   0.63   

                                                                               \
app_name    shiksha mitra shiksha.com simplilearn stack overflow swayam toppr   
ltr_segment                                                                     
HH                   0.04        0.16        1.21           0.01   3.30  0.01   
LTR 0                0.04        0.15        1.10            NaN   2.76   NaN   
PHH                  0.03        0.17        2.14           0.02   4.31  0.01   

                                                           
app_name     udemy unacademy vedantu vidyakul whitehat jr  
ltr_segment                                                
HH           10.56     12.25    0.54     0.10        0.29  
LTR 0         7.24     11.92    0.63     0.20        0.33  
PHH          20.69     12.01    0.39     0.06        0.20

In [54]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        22856                   26494
LTR 0                      4464                    4770
PHH                       64996                   96002

In [55]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        53.87                   62.44
LTR 0                     56.00                   59.84
PHH                       47.16                   69.66

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to move towards PHH.
    - Finance_Investment, Office, Ride haling, Educational, Gaming
    - Rest
- Whenever an app belongs to the app-categories listed below, customers are unlikely to move towards PHH.
    - Driver_App

### Other

#### Service Affinity

In [56]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,19287,2.02
1,LINK_AUTO,52773,5.52
2,LINK_CAB,24384,2.55
3,NO_AFFINITY,86489,9.04
4,ONLY_AUTO,127510,13.33
5,ONLY_CAB,74886,7.83
6,ONLY_LINK,495426,51.80
7,UNKNOWN,75658,7.91


In [57]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers'])

customers                                                \
categories_l2    Driver_App Educational Finance_Investment Gaming  Office   
service_affinity                                                            
AUTO_CAB               1023        4587               7222   3635   11630   
LINK_AUTO              3731       11659              20849  11139   29078   
LINK_CAB               2727        3968               8933   5216   11570   
NO_AFFINITY            7013       17364              33944  18379   46916   
ONLY_AUTO              6931       31318              47084  25148   75252   
ONLY_CAB               5798       14052              26714  14431   39160   
ONLY_LINK             46305       91840             188857  99975  254710   

                                      
categories_l2       Rest Ride haling  
service_affinity                      
AUTO_CAB           19280       14874  
LINK_AUTO          52758       37191  
LINK_CAB           24378       16618  
NO_AFFINITY        86460       65150  
ONLY_AUTO         127463       94121  
ONLY_CAB           74833       53278  
ONLY_LINK         495204      328341

In [58]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                5.30       23.78              37.44  18.85  60.30   
LINK_AUTO               7.07       22.09              39.51  21.11  55.10   
LINK_CAB               11.18       16.27              36.63  21.39  47.45   
NO_AFFINITY             8.11       20.08              39.25  21.25  54.25   
ONLY_AUTO               5.44       24.56              36.93  19.72  59.02   
ONLY_CAB                7.74       18.76              35.67  19.27  52.29   
ONLY_LINK               9.35       18.54              38.12  20.18  51.41   

                                     
categories_l2      Rest Ride haling  
service_affinity                     
AUTO_CAB          99.96       77.12  
LINK_AUTO         99.97       70.47  
LINK_CAB          99.98       68.15  
NO_AFFINITY       99.97       75.33  
ONLY_AUTO         99.96       73.81  
ONLY_CAB          99.93       71.15  
ONLY_LINK         99.96       66.27

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [59]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,252480,26.40
1,MEDIUM_DISTANCE,234080,24.47
2,SHORT_DISTANCE,226947,23.73
3,UNKNOWN,242906,25.40


In [60]:
df1 = df_customer_data_explode[~df_customer_data_explode['distance_preference'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,distance_affinity_city[['distance_preference','customers']], how= 'left', on='distance_preference')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2        Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                            
LONG_DISTANCE              9.94       16.02              35.29  20.48  47.14   
MEDIUM_DISTANCE            7.33       19.97              38.23  19.93  54.00   
SHORT_DISTANCE             6.67       24.20              39.78  19.68  58.76   

                                        customers              \
categories_l2         Rest Ride haling Driver_App Educational   
distance_preference                                             
LONG_DISTANCE        99.95       66.46    25107.0     40453.0   
MEDIUM_DISTANCE      99.96       69.50    17161.0     46751.0   
SHORT_DISTANCE       99.95       71.28    15135.0     54931.0   

                                                                     \
categories_l2       Finance_Investment   Gaming    Office      Rest   
distance_preference                                                   
LONG_DISTANCE                  89097.0  51720.0  119031.0  252359.0   
MEDIUM_DISTANCE                89499.0  46644.0  126411.0  233993.0   
SHORT_DISTANCE                 90280.0  44669.0  133362.0  226831.0   

                                 
categories_l2       Ride haling  
distance_preference              
LONG_DISTANCE          167797.0  
MEDIUM_DISTANCE        162680.0  
SHORT_DISTANCE         161765.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [61]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,203128,21.24
1,MALE,725103,75.81
2,OTHERS,2716,0.28
3,UNKNOWN,25466,2.66


In [62]:
df1 = df_customer_data_explode[~df_customer_data_explode['gender'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,gender_city[['gender','customers']], how= 'left', on='gender')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
gender                                                                          
FEMALE               3.54       24.31              28.85  22.02  58.73  99.95   
MALE                10.03       18.33              39.74  19.73  50.57  99.95   
OTHERS               7.22       13.11              15.80  22.39  32.70  99.96   

                           customers                                           \
categories_l2 Ride haling Driver_App Educational Finance_Investment    Gaming   
gender                                                                          
FEMALE              73.14     7187.0     49390.0            58607.0   44737.0   
MALE                66.84    72706.0    132902.0           288121.0  143076.0   
OTHERS              51.84      196.0       356.0              429.0     608.0   

                                               
categories_l2    Office      Rest Ride haling  
gender                                         
FEMALE         119294.0  203035.0    148561.0  
MALE           366681.0  724772.0    484649.0  
OTHERS            888.0    2715.0      1408.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [63]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,347397,36.32
1,LOW_INCOME,60686,6.35
2,MEDIUM_INCOME,367712,38.45
3,UNKNOWN,180618,18.88


In [64]:
df1 = df_customer_data_explode[~df_customer_data_explode['income_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,income_segment_city[['income_segment','customers']], how= 'left', on='income_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2   Driver_App Educational Finance_Investment Gaming Office   
income_segment                                                            
HIGH_INCOME           7.79       23.86              47.22  22.97  63.75   
LOW_INCOME            7.73       10.71              19.53  12.98  30.90   
MEDIUM_INCOME         8.84       19.63              33.27  19.60  50.55   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
income_segment                                                                
HIGH_INCOME     99.98       79.56    27051.0     82891.0           164050.0   
LOW_INCOME      99.85       45.08     4689.0      6500.0            11851.0   
MEDIUM_INCOME   99.95       65.16    32514.0     72164.0           122355.0   

                                                         
categories_l2    Gaming    Office      Rest Ride haling  
income_segment                                           
HIGH_INCOME     79792.0  221476.0  347317.0    276392.0  
LOW_INCOME       7880.0   18754.0   60592.0     27357.0  
MEDIUM_INCOME   72084.0  185884.0  367543.0    239611.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [65]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,291938,30.52
1,PS,255830,26.75
2,UNKNOWN,408645,42.73


In [66]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                      8.04       19.86              39.07  20.61  55.27   
PS                       7.65       21.43              39.77  20.17  56.29   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.97       73.36    23486.0     57985.0   
PS                 99.96       72.42    19566.0     54812.0   

                                                                               
categories_l2     Finance_Investment   Gaming    Office      Rest Ride haling  
price_sensitivity                                                              
NPS                         114066.0  60173.0  161351.0  291841.0    214158.0  
PS                          101753.0  51591.0  144007.0  255733.0    185273.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [67]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,204073,21.34
1,b. LOW,373611,39.08
2,c. MEDIUM,221889,23.21
3,d. HIGH,156552,16.37


In [68]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO             10.93       18.91              34.97  21.37  47.70  99.95   
b. LOW               8.91       19.12              36.66  20.27  50.46  99.95   
c. MEDIUM            7.52       20.30              38.22  19.99  54.38  99.96   
d. HIGH              5.81       21.19              40.45  19.18  60.13  99.95   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
rpc_bucket                                                                     
a. ZERO             67.26    22306.0     38594.0            71362.0  43615.0   
b. LOW              64.98    33275.0     71437.0           136957.0  75740.0   
c. MEDIUM           68.89    16675.0     45054.0            84800.0  44345.0   
d. HIGH             76.42     9092.0     33171.0            63323.0  30024.0   

                                               
categories_l2    Office      Rest Ride haling  
rpc_bucket                                     
a. ZERO         97336.0  203976.0    137265.0  
b. LOW         188522.0  373435.0    242788.0  
c. MEDIUM      120663.0  221798.0    152849.0  
d. HIGH         94132.0  156481.0    119633.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [69]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,841000,87.93
1,11-15,25889,2.71
2,6-10,77335,8.09
3,Above 15,12189,1.27


In [70]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   92.06       86.98              86.79  88.67  85.94   
11-15                  1.62        2.94               3.03   2.46   3.22   
6-10                   5.54        8.85               8.85   7.69   9.36   
Above 15               0.77        1.23               1.34   1.17   1.47   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               87.93       86.29  
11-15              2.71        3.16  
6-10               8.09        9.06  
Above 15           1.27        1.49

In [71]:
df_customer_data.app_count.describe()

count    956413.000000
mean         17.726033
std           9.052043
min           1.000000
25%          11.000000
50%          16.000000
75%          23.000000
max          93.000000
Name: app_count, dtype: float64

#### Category Count Bucket

In [72]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,115636,12.09
1,2-3,481428,50.34
2,Above 3,359349,37.57


In [73]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.07        0.01               0.01   0.01   0.01   
2-3                        41.89       21.79              29.31  34.85  37.74   
Above 3                    58.04       78.21              70.68  65.15  62.24   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      12.06        0.02  
2-3                    50.35       47.52  
Above 3                37.59       52.46

In [74]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2
0,"{['Ride haling', 'Office', 'Rest']}"
1,"{['Educational', 'Ride haling', 'Rest']}"
2,"{['Ride haling', 'Office', 'Rest']}"
3,"{['Ride haling', 'Office', 'Rest']}"
4,"{['Ride haling', 'Office', 'Rest']}"
...,...
481423,"{['Finance_Investment', 'Rest']}"
481424,"{['Gaming', 'Rest']}"
481425,"{['Ride haling', 'Office', 'Rest']}"
481426,"{['Ride haling', 'Rest']}"


In [75]:
df_customer_data.category_count.describe()

count    956413.000000
mean          3.063021
std           1.264304
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

#### Customer Use-Case 

In [76]:
usecase_tag_city = df_customer_data\
                        .groupby(['usecase_tag'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
usecase_tag_city['% city threshold']=(usecase_tag_city['customers']*100.00/usecase_tag_city.customers.sum()).round(2)
usecase_tag_city.sort_values(by=['usecase_tag'],ascending=[True])

,usecase_tag,customers,% city threshold
0,Unknown,447862,46.83
1,educational,23643,2.47
2,food,12147,1.27
3,govt_amenity,3675,0.38
4,health_and_personal,25876,2.71
5,household_needs,20857,2.18
6,leisure,45715,4.78
7,office,27205,2.84
8,place_of_worship,8150,0.85
9,residential,205889,21.53


In [77]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'usecase_tag'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'usecase_tag'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='usecase_tag' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2       Driver_App Educational Finance_Investment Gaming Office   
usecase_tag                                                                   
Unknown                  45.03       48.11              49.99  46.71  50.02   
educational               2.05        3.31               2.27   2.29   2.53   
food                      1.13        1.27               1.19   1.24   1.25   
govt_amenity              0.42        0.35               0.36   0.40   0.36   
health_and_personal       2.88        2.41               2.43   2.68   2.41   
household_needs           2.24        2.08               1.98   2.18   2.00   
leisure                   4.49        4.84               4.66   4.73   4.76   
office                    3.19        2.56               2.87   2.74   2.78   
place_of_worship          0.86        0.81               0.75   0.88   0.76   
residential              23.57       20.19              19.29  21.83  19.91   
transit_station          14.15       14.06              14.19  14.31  13.21   

                                        
categories_l2         Rest Ride haling  
usecase_tag                             
Unknown              46.83       46.76  
educational           2.47        2.58  
food                  1.27        1.33  
govt_amenity          0.38        0.37  
health_and_personal   2.71        2.74  
household_needs       2.18        2.20  
leisure               4.78        4.99  
office                2.85        2.92  
place_of_worship      0.85        0.83  
residential          21.53       21.89  
transit_station      14.16       13.40

#### AO-NET Funnel

In [78]:
df_customer_data['city'] = 'Delhi, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Delhi, Android"
% fe2rr,43.84
% g2n,57.32
% fe2net,25.13


In [79]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Ride haling,Office,Finance_Investment,Gaming,Educational,Driver_App
% fe2rr,43.84,45.02,44.6,43.79,44.33,43.7,43.44
% g2n,57.32,57.88,59.79,59.38,55.98,59.13,49.91
% fe2net,25.13,26.05,26.67,26.0,24.82,25.84,21.68
